# Vetorização e armazenamento no Qdrant
Para fazer a ingestão de dados com o **Notebook 01**, configuramos o Ollama para gerar os embeddings (vetores) e o Qdrant para armazená-los.

A base de conhecimento simula o que acontece em qualquer empresa: **as regras mudam com o tempo**. Vamos importar dois documentos em datas diferentes:

| Importação | Arquivo | Conteúdo |
|---|---|---|
| 2021-03-10 | `politica_de_trocas_e_devolucoes.txt` | Política completa (8 seções) |
| 2024-03-01 | `atualizacao_politica_2024.txt` | Atualização **parcial**: só as regras de Eletrônicos e de Suplementos mudaram |

A data de importação fica **apenas no metadado (payload)** do Qdrant, não no texto. É assim que acontece em pipelines reais e é exatamente por isso que o LLM, sozinho, não consegue saber qual versão é a vigente.

Como estamos rodando o Jupyter Notebook na máquina local (fora da rede do Docker), nos conectamos aos serviços usando o localhost.

## 1. Instalação das dependências

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Script de Ingestão (LangChain)

In [2]:
import re
import uuid
import ollama
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBEDDING_MODEL = "nomic-embed-text-v2-moe"

### 2.1. Configuramos o cliente do Qdrant

In [3]:
qdrant_url = "http://localhost:6333"
collection_name = "portfolio_agentic_rag"
client = QdrantClient(url=qdrant_url)

# A dimensão do vetor vem do próprio modelo, evitando um número mágico fixo
dimensao = len(ollama.embed(model=EMBEDDING_MODEL, input="teste")["embeddings"][0])

# Recriamos a coleção para permitir rodar o notebook várias vezes do zero
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dimensao, distance=Distance.COSINE),
)

True

### 2.2. Histórico de importações
Cada arquivo carrega a data em que entrou na base. A atualização de 2024 **não substitui o arquivo inteiro**: ela só traz as seções que mudaram. Todas as outras regras de 2021 continuam valendo.

In [4]:
IMPORTACOES = [
    {"arquivo": "politica_de_trocas_e_devolucoes.txt", "data_importacao": "2021-03-10"},
    {"arquivo": "atualizacao_politica_2024.txt", "data_importacao": "2024-03-01"},
]

### 2.3. Quebrar os textos em pedaços menores (Chunks)

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

def gerar_chunks(texto: str) -> list[str]:
    # Primeiro separamos por seção (linha em branco) para que cada chunk contenha uma única regra.
    # O splitter só entra em ação se alguma seção passar de 1000 caracteres.
    secoes = re.split(r"\n\s*\n", texto.strip())
    chunks = []
    for secao in secoes:
        chunks.extend(text_splitter.split_text(secao))
    return chunks

## A Decisão Arquitetural do Chunking (1 seção = 1 chunk)
Diferente do RAG tradicional, onde blocos maiores às vezes são utilizados para "inundar" o LLM de contexto, nossa arquitetura exige precisão cirúrgica: **cada chunk deve conter exatamente uma regra**.

* **Precisão na Triagem (Agente Laya):** O modelo `laya:322m` classifica *qual regra* cada chunk define (Eletrônicos, Móveis, Suplementos...). Se um chunk misturar duas seções, ele não pode ser atribuído a uma única regra, e o descarte de versões obsoletas deixa de funcionar.

* **Por que não só `chunk_size=1000`?** Aplicado direto no texto inteiro, o `RecursiveCharacterTextSplitter` junta seções curtas no mesmo bloco ou separa o título de uma seção do seu corpo.

* **Maximização do Signal-to-Noise Ratio:** Modelos de linguagem sofrem do efeito *Lost in the Middle*. Chunks curtos e filtrados garantem que o Llama 3.1 receba um contexto pequeno e denso.

* **Prevenção de Ruptura Semântica (Overlap 100):** Se uma seção ultrapassar 1000 caracteres, o `chunk_overlap=100` garante que o sujeito de uma frase não fique no Chunk A enquanto a condição da regra cai no Chunk B.

### 2.4. Ingerir os dados no Qdrant
O `nomic-embed-text-v2-moe` foi treinado com **prefixos de tarefa**: `search_document:` para os textos indexados e `search_query:` para as perguntas (usado nos notebooks 02 e 03). Sem eles, a qualidade do retrieval cai. Também usamos `ollama.embed`, que vetoriza todos os chunks de um arquivo em uma única chamada.

In [6]:
for importacao in IMPORTACOES:
    with open(f"../data/{importacao['arquivo']}", "r", encoding="utf-8") as arquivo:
        chunks = gerar_chunks(arquivo.read())

    vetores = ollama.embed(
        model=EMBEDDING_MODEL,
        input=[f"search_document: {chunk}" for chunk in chunks]
    )["embeddings"]

    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vetor,
            payload={
                "texto": chunk,
                "origem": importacao["arquivo"],
                "data_importacao": importacao["data_importacao"],
                "chunk_index": i
            }
        )
        for i, (chunk, vetor) in enumerate(zip(chunks, vetores))
    ]

    client.upsert(collection_name=collection_name, wait=True, points=points)
    print(f"{importacao['data_importacao']} | {importacao['arquivo']}: {len(points)} chunks armazenados")

2021-03-10 | politica_de_trocas_e_devolucoes.txt: 8 chunks armazenados


2024-03-01 | atualizacao_politica_2024.txt: 2 chunks armazenados


### 2.5. Conferir o conteúdo da coleção

In [7]:
pontos, _ = client.scroll(collection_name=collection_name, limit=100, with_payload=True)
for ponto in sorted(pontos, key=lambda p: (p.payload["data_importacao"], p.payload["chunk_index"])):
    print(f"{ponto.payload['data_importacao']} | {ponto.payload['texto'][:70]}")

2021-03-10 | 1. Visão Geral e Nota Fiscal
Nossa política de trocas, devoluções e ar
2021-03-10 | 2. Regras para Roupas, Calçados e Vestuário
O cliente tem até 30 (trin
2021-03-10 | 3. Regras para Móveis e Decoração
O prazo de arrependimento para móvei
2021-03-10 | 4. Regras para Brinquedos e Artigos Infantis
Brinquedos possuem prazo 
2021-03-10 | 5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulare
2021-03-10 | 6. Regras para Eletrodomésticos da Linha Branca
Para geladeiras, fogõe
2021-03-10 | 7. Regras para Artigos Esportivos e Suplementos
Equipamentos de ginást
2021-03-10 | 8. Análise Técnica e Reembolso
O produto devolvido passa por análise t
2024-03-01 | 5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulare
2024-03-01 | 7. Regras para Artigos Esportivos e Suplementos
Equipamentos de ginást
